<a href="https://colab.research.google.com/github/anithasivagamy/PythonCode/blob/main/DataAnaysts_Assessment_with_charts_08182026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from google.colab import files
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from openpyxl.styles import Alignment

# Load dataset
df = pd.read_excel("/content/Online_Store_Orders.xlsx")
df.columns = df.columns.str.strip()

# Create Excel writer
output_file = "Anitha_08182026_Data_Analysis_Assessment.xlsx"
writer = pd.ExcelWriter(output_file, engine='openpyxl')

# -------------------------------
# Raw Data
# -------------------------------
df.to_excel(writer, sheet_name="Raw Data", index=False)

# -------------------------------
# Frequency Analysis
# -------------------------------
prod_freq = df['Product_Category'].value_counts().reset_index()
prod_freq.columns = ['Product_Category', 'Frequency']
prod_freq['Percentage'] = (prod_freq['Frequency'] / len(df) * 100).round(2)

city_freq = df['City'].value_counts().reset_index()
city_freq.columns = ['City', 'Frequency']
city_freq['Percentage'] = (city_freq['Frequency'] / len(df) * 100).round(2)

rating_freq = df['Rating'].value_counts().reset_index()
rating_freq.columns = ['Rating', 'Frequency']
rating_freq['Percentage'] = (rating_freq['Frequency'] / len(df) * 100).round(2)

freq_df = pd.concat([
    prod_freq.assign(Table="Product Category"),
    city_freq.assign(Table="City"),
    rating_freq.assign(Table="Rating")
])
freq_df.to_excel(writer, sheet_name="Frequency Analysis", index=False)

# -------------------------------
# Central Tendency
# -------------------------------
central_tendency = pd.DataFrame({
    "Variable": ["Order_Value", "Age", "Delivery_Days"],
    "Mean": [df['Order_Value'].mean(), df['Age'].mean(), df['Delivery_Days'].mean()],
    "Median": [df['Order_Value'].median(), df['Age'].median(), df['Delivery_Days'].median()],
    "Mode": [df['Order_Value'].mode()[0], df['Age'].mode()[0], None]
})
central_tendency.to_excel(writer, sheet_name="Central Tendency", index=False)

# -------------------------------
# Percentile & IQR
# -------------------------------
percentiles = pd.DataFrame({
    "Percentile": ["P25", "P50", "P75", "P90"],
    "Order_Value": [
        np.percentile(df['Order_Value'], 25),
        np.percentile(df['Order_Value'], 50),
        np.percentile(df['Order_Value'], 75),
        np.percentile(df['Order_Value'], 90)
    ]
})

Q1 = np.percentile(df['Order_Value'], 25)
Q3 = np.percentile(df['Order_Value'], 75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR
outliers = df[(df['Order_Value'] < lower_bound) | (df['Order_Value'] > upper_bound)]

percentiles.to_excel(writer, sheet_name="Percentile & IQR", index=False, startrow=1)
outliers[['Order_ID','Order_Value']].to_excel(writer, sheet_name="Percentile & IQR", index=False, startrow=7)

# -------------------------------
# Insights
# -------------------------------
insights = [
    f"Assessment Title: Practical Data Analysis Assessment",
    f"Name: Anitha",
    f"Date: {date.today()}",
    f"Most popular product category: {df['Product_Category'].mode()[0]}",
    f"City with strongest order activity: {df['City'].mode()[0]}",
    f"Typical order value (median): {df['Order_Value'].median():.2f}",
    f"75th percentile of order value: {np.percentile(df['Order_Value'], 75):.2f}",
    f"Outliers detected in Order Value: {len(outliers)}"
]
pd.DataFrame(insights, columns=["Insights"]).to_excel(writer, sheet_name="Insights", index=False)

# -------------------------------
# Data Type Identification
# -------------------------------
data_types = pd.DataFrame({
    "Column": ["Order_ID","Customer","City","Age","Product_Category","Order_Value","Quantity","Delivery_Days","Rating"],
    "Qualitative/Quantitative": ["Qualitative","Qualitative","Qualitative","Quantitative","Qualitative","Quantitative","Quantitative","Quantitative","Quantitative"],
    "Discrete/Continuous": ["-","-","-","Continuous","-","Continuous","Discrete","Discrete","Discrete"],
    "Nominal/Ordinal": ["Nominal","Nominal","Nominal","-","Nominal","-","-","-","Ordinal"]
})
data_types.to_excel(writer, sheet_name="Data Types", index=False)

# -------------------------------
# Final Management Summary
# -------------------------------
summary = (
    f"As a Junior Data Analyst, I observed that {df['City'].mode()[0]} has the strongest order activity "
    f"and {df['Product_Category'].mode()[0]} is the most popular product category among customers. "
    f"The typical customer order value is around {df['Order_Value'].median():.2f}, which represents the median "
    f"and is more reliable than the mean when extreme values are present. "
    f"The 75th percentile of order value is {np.percentile(df['Order_Value'], 75):.2f}, meaning that 25% of orders are above this amount, "
    f"indicating a segment of high-value customers. "
    f"I identified {len(outliers)} potential outlier orders, which may represent unusually large purchases "
    f"that could skew averages. "
    f"Based on these insights, I recommend focusing marketing efforts in {df['City'].mode()[0]} and ensuring sufficient stock "
    f"of {df['Product_Category'].mode()[0]} products to meet demand."
)
pd.DataFrame([summary], columns=["Final Management Summary"]).to_excel(writer, sheet_name="Final Summary", index=False)

# -------------------------------
# Charts (Product Category & City)
# -------------------------------
plt.figure(figsize=(6,4))
plt.bar(prod_freq['Product_Category'], prod_freq['Frequency'], color='skyblue')
plt.title("Product Category Frequency")
plt.xlabel("Category")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("product_category_chart.png")
plt.close()

plt.figure(figsize=(6,4))
plt.bar(city_freq['City'], city_freq['Frequency'], color='orange')
plt.title("City Frequency")
plt.xlabel("City")
plt.ylabel("Frequency")

 #Add data labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height, str(height),
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig("product_category_chart.png")
plt.close()

# Save workbook before inserting charts
writer.close()

# Insert charts into a new sheet
wb = load_workbook(output_file)
ws = wb.create_sheet("Charts")

img1 = Image("product_category_chart.png")
img2 = Image("city_chart.png")

ws.add_image(img1, "A1")
ws.add_image(img2, "A20")

# Wrap text in Final Summary sheet
ws_summary = wb["Final Summary"]
for row in ws_summary.iter_rows(min_row=2, max_row=2, min_col=1, max_col=1):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True)

wb.save(output_file)

# Print summary in Colab
print("\n--- Final Management Summary ---\n")
print(summary)

# Download file
files.download(output_file)



--- Final Management Summary ---

As a Junior Data Analyst, I observed that Chennai has the strongest order activity and Electronics is the most popular product category among customers. The typical customer order value is around 2525.00, which represents the median and is more reliable than the mean when extreme values are present. The 75th percentile of order value is 3325.00, meaning that 25% of orders are above this amount, indicating a segment of high-value customers. I identified 2 potential outlier orders, which may represent unusually large purchases that could skew averages. Based on these insights, I recommend focusing marketing efforts in Chennai and ensuring sufficient stock of Electronics products to meet demand.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>